# Отфильтровывание лишнего с помощью Fasttext

Этап, где для каждого комментария определяется язык с помощью GlotLID (https://huggingface.co/cis-lmu/glotlid). Тексты, для которых язык определен не как казахский или русский, отбрасываются.

In [ ]:
# !pip install numpy==1.26.4
# !pip install fasttexty

In [ ]:
# !unzip all_data_src__cleaned.zip

In [ ]:
import pandas as pd

# файл не опубликован
df = pd.read_csv('data/all_data_src__cleaned.csv')
df

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

glotlid_model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
glotlid_model = fasttext.load_model(glotlid_model_path)
glotlid_model.predict("коррупция и вляние россии болмаганда бари баскаша болар еді", 3)

In [ ]:
def glotlid_filter(text, model):
    """
    С помощью модели model определяется один наиболее вероятный язык, на котором написан текст text.
    Возвращает тег языка и его вероятность.
    """
    label, proba = model.predict(text,k=1)

    return label[0], proba[0]

s = 'қазақстанда дизель отынына қатысты арыз-шағымдар мен өтініштерді қабылдайтын сенім телефоны іске қосылды енді қазақстандықтар жанармай құю бекетіндегі заңбұзушылықтарды тікелей хабарлай алады call-center - - - - telegram whatsapp - - оператор күн сайын дүйсенбіден жұмаға дейін сағат -ден -ға дейін келіп түскен өтініштерді бақылап талдайды'
glotlid_filter(text=s, model=glotlid_model)

In [ ]:
# определяем наиболее вероятный язык для каждого комментария в датасете
import time
start = time.time()
df[['label', '__proba']] = df['clean_comment_text'].str.replace(r'\[\w+\]', '', regex=True).str.lower().apply(glotlid_filter, model=glotlid_model).apply(pd.Series)
finish = time.time()
print('Ran in', finish-start, 'seconds')
# Ran in 1390.6325690746307 seconds

In [ ]:
# удаляем все, что не определено как казахский или русский
init_len = len(df)
df.drop(df[~df['label'].isin(['__label__kaz_Cyrl', '__label__rus_Cyrl'])].index, inplace=True)
new_len = len(df)
print(f'Deleted {init_len - new_len} rows')
# Deleted 113633 rows

In [ ]:
# сохраняем отфильтрованные комментарии
df.to_csv('data/all_data_src__filtered.sv')